# 03 - Transcription (faster-whisper)

ASR converts speech to text so that grammar-related cues can be measured. Transcripts are **cached** in `data/processed/transcripts/`; the run is resumable and one failing file never aborts the batch.

> Caveat: Whisper is a strong language model and tends to produce fluent text, which can silently *repair* some grammatical errors. Text features therefore under-estimate certain errors; acoustic features and ASR-level cues (disfluencies, empty output) partly compensate.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)

In [ ]:
import yaml; print(yaml.safe_dump(cfg["asr"]))
RUN_ASR = False   # True = transcribe from here (slow on CPU: use `python scripts/transcribe.py` instead)
LIMIT = None      # e.g. 5 for a quick smoke test

In [ ]:
if RUN_ASR and ctx is not None:
    from src.pipeline import run_transcription
    for split in ("train", "test"):
        run_transcription(ctx, split, manifest="test", limit=LIMIT)

## Inspect cached transcripts

In [ ]:
tp = ctx.processed("transcripts") / "train_transcripts.csv" if ctx else None
if tp is not None and tp.exists():
    tr = pd.read_csv(tp); tr["transcript"] = tr["transcript"].fillna("")
    tr = tr.merge(ctx.meta().train, on="filename", how="left")
    print("status:", tr["status"].value_counts().to_dict(), "| ASR models:", tr["asr_model"].unique())
    print("empty transcripts:", int((tr["transcript"].str.strip() == "").sum()))
    tr["n_words"] = tr["transcript"].str.split().str.len()
    display(tr.groupby("label")["n_words"].agg(["count", "mean", "median"]).round(1))
    plt.figure(figsize=(7, 4)); plt.scatter(tr["label"] + np.random.normal(0, .04, len(tr)), tr["n_words"], s=8, alpha=.4)
    plt.xlabel("label"); plt.ylabel("words in transcript"); plt.show()
    for lab in sorted(tr["label"].dropna().unique())[:: max(1, len(tr["label"].unique()) // 4)]:
        r = tr[tr["label"] == lab].iloc[0]; print(f"\n--- label {lab} | {r.filename}\n{r.transcript[:400]}")
else:
    print(PENDING)

## Failed files (if any)

In [ ]:
if tp is not None and tp.exists():
    display(tr[tr["status"] != "ok"][["filename", "error_message"]])
    print("Retry with: python scripts/transcribe.py --retry-failed")